# PyTorch: Transfer Learning

In [ ]:
import torch
from torchinfo import summary
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchmetrics.classification import MulticlassAccuracy
from common import CV_DATASETS_DIR
import common.torch as ct

In [ ]:
ct.set_default_seed()
ct.set_default_optimizations()

In [ ]:
device = ct.get_optimal_device()

In [ ]:
print(f"PyTorch: version {torch.__version__}")
print(f"PyTorch: {device.type.upper()} device")

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
N_EPOCHS = 5
# Other parameters
IMAGE_SIZE = (224,224)

## Prepare Datasets

In [ ]:
DATASET_PATH = CV_DATASETS_DIR/"animals"

tr_dataset = datasets.OxfordIIITPet(root=DATASET_PATH,
                                    transform=EfficientNet_B0_Weights.DEFAULT.transforms(),
                                    split="trainval",
                                    download=True)

ts_dataset = datasets.OxfordIIITPet(root=DATASET_PATH,
                                    transform=EfficientNet_B0_Weights.DEFAULT.transforms(),
                                    split="test",
                                    download=True)

assert tr_dataset.class_to_idx == ts_dataset.class_to_idx, "Train and Test class indices mismatch!"

In [ ]:
tr_dl = DataLoader(tr_dataset, batch_size=BATCH_SIZE, num_workers=2, shuffle=True)
ts_dl = DataLoader(ts_dataset, batch_size=BATCH_SIZE, num_workers=2)
len(tr_dl), len(ts_dl)

In [ ]:
n_classes = len(tr_dataset.classes)
n_classes

## Define Model

In [ ]:
# Create a model using pretrained weights
model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT).to(device)

# Create classifier layer
model.classifier = nn.Sequential(
    nn.Linear(in_features=1280, out_features=512),
    nn.BatchNorm1d(num_features=512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(in_features=512, out_features=n_classes),
).to(device)

# Freeze features layers
for param in model.features.parameters():
    param.requires_grad = False

In [ ]:
summary(model=model,
        input_size=(1, 3)+IMAGE_SIZE,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

## Train Model

In [ ]:
logs_dir, writer = ct.get_summary_writer("pt_transfer_learning", "efficientnet_b0", "5_epochs")
logs_dir

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
optimizer = optim.AdamW(params=model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
accuracy = MulticlassAccuracy(num_classes=n_classes).to(device)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-7)

In [ ]:
ct.train(model=model,
         tr_dl=tr_dl,
         ts_dl=ts_dl,
         optimizer=optimizer,
         criterion=criterion,
         metric=accuracy,
         n_epochs=N_EPOCHS,
         writer=writer,
         device=device,
         scheduler=scheduler)